In [ ]:
"""
DeepEcoHAB / EcoHAB RFID dataframe reformatter GUI.

Run:
    pip install PySide6 pandas
    python deepecohab_rfid_reformatter_gui.py

Supported input DAQ formats
---------------------------
1. New DeepEcoHAB RFID DAQ
   antenna ; animal_id ; hexadecimal_board_time ; ISO_datetime

2. Old EcoHAB RFID DAQ + new firmware
   CSV with header: PC Time, Channel, Board Time, RFID Tag

3. Old EcoHAB RFID DAQ + old firmware
   tab-separated, already-compressed old format:
   index, date, time, antenna, time_under, animal_id

Output format
-------------
Tab-separated old EcoHAB-compatible format without header:
index, date, time, antenna, time_under, animal_id
"""

from __future__ import annotations

import json
import math
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from PySide6.QtCore import QObject, Qt, QThread, Signal, Slot
    from PySide6.QtGui import QColor, QPalette
    from PySide6.QtWidgets import (
        QApplication,
        QCheckBox,
        QFileDialog,
        QFormLayout,
        QGroupBox,
        QGridLayout,
        QHBoxLayout,
        QLabel,
        QLineEdit,
        QMainWindow,
        QMessageBox,
        QPushButton,
        QPlainTextEdit,
        QProgressBar,
        QSpinBox,
        QSizePolicy,
        QVBoxLayout,
        QWidget,
    )
except ImportError as exc:  # pragma: no cover - shown only when run without PySide6
    print("Missing dependency: PySide6. Install it with: pip install PySide6 pandas")
    raise exc


# -----------------------------
# Format names
# -----------------------------
NEW_DEEPECOHAB = "New DeepEcoHAB RFID DAQ"
OLD_ECOHAB_NEW_FW = "Old EcoHAB RFID DAQ + new firmware"
OLD_ECOHAB_OLD_FW = "Old EcoHAB RFID DAQ + old firmware"
UNKNOWN_FORMAT = "Unknown / unsupported"

SUPPORTED_EXTENSIONS = {".csv", ".txt", ".tsv"}
HEX_TAG_RE = re.compile(r"^[0-9A-Fa-f]{6,16}$")


@dataclass
class DetectedFile:
    path: Path
    fmt: str
    reason: str = ""


@dataclass
class ReformatSettings:
    input_dir: Path
    output_dir: Path
    stitch_files: bool
    keep_only_listed_tags: bool
    tag_numbers: List[str]
    rename_antennas: bool
    antenna_map: Dict[int, int]
    separation_window_ms: int


# -----------------------------
# Generic helpers
# -----------------------------
def normalize_tag(value: object) -> str:
    """Normalize a tag while preserving leading zeros."""
    if value is None:
        return ""
    text = str(value).strip().strip('"').strip("'").upper()
    return text


def unique_preserve_order(values: Iterable[str]) -> List[str]:
    seen = set()
    out: List[str] = []
    for value in values:
        value = normalize_tag(value)
        if not value:
            continue
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out


def first_non_empty_lines(path: Path, n: int = 5) -> List[str]:
    lines: List[str] = []
    try:
        with path.open("r", encoding="utf-8-sig", errors="replace") as handle:
            for line in handle:
                stripped = line.strip()
                if stripped:
                    lines.append(stripped)
                    if len(lines) >= n:
                        break
    except OSError:
        return []
    return lines


def looks_like_int(text: str) -> bool:
    try:
        int(str(text).strip())
        return True
    except Exception:
        return False


def looks_like_float(text: str) -> bool:
    try:
        float(str(text).strip())
        return True
    except Exception:
        return False


def normalize_pc_time_text(series: pd.Series) -> pd.Series:
    """Convert timestamps like 2025-11-14 11:14:52:496 to ...52.496."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"(\d{2}:\d{2}:\d{2}):(\d+)$", r"\1.\2", regex=True)
    )


def to_datetime_safe(series: pd.Series) -> pd.Series:
    return pd.to_datetime(normalize_pc_time_text(series), errors="coerce")


def replace_antenna_values(series: pd.Series, mapping: Dict[int, int]) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").astype("Int64")
    if not mapping:
        return numeric

    def repl(value: object) -> object:
        if pd.isna(value):
            return pd.NA
        return mapping.get(int(value), int(value))

    return numeric.map(repl).astype("Int64")


def apply_tag_filter(df: pd.DataFrame, tags: Sequence[str], keep_only: bool) -> pd.DataFrame:
    df = df.copy()
    df["animal_id"] = df["animal_id"].map(normalize_tag)
    if keep_only:
        tag_set = set(normalize_tag(tag) for tag in tags if normalize_tag(tag))
        df = df[df["animal_id"].isin(tag_set)].copy()
    return df


def parse_output_datetime(df: pd.DataFrame) -> pd.Series:
    if "datetime" in df.columns:
        dt = pd.to_datetime(df["datetime"], errors="coerce")
    else:
        dt = pd.Series(pd.NaT, index=df.index)

    if (dt.isna().any()) and {"date", "time"}.issubset(df.columns):
        fallback = df["date"].astype(str).str.replace(".", "-", regex=False) + " " + df[
            "time"
        ].astype(str)
        fallback_dt = to_datetime_safe(fallback)
        dt = dt.fillna(fallback_dt)
    return dt


def finalize_old_style_output(df: pd.DataFrame, sort_by_datetime: bool = True) -> pd.DataFrame:
    """Build final six-column old EcoHAB-compatible output."""
    if df.empty:
        return pd.DataFrame(columns=["index", "date", "time", "antenna", "time_under", "animal_id"])

    out = df.copy()
    out["datetime"] = parse_output_datetime(out)
    if sort_by_datetime:
        out = out.sort_values("datetime", kind="mergesort", na_position="last").reset_index(drop=True)
    else:
        out = out.reset_index(drop=True)

    dt = pd.to_datetime(out["datetime"], errors="coerce")
    date_from_dt = dt.dt.strftime("%Y.%m.%d")
    time_from_dt = dt.dt.strftime("%H:%M:%S.%f").str[:-3]

    if "date" in out.columns:
        date = date_from_dt.fillna(out["date"].astype(str))
    else:
        date = date_from_dt.fillna("")

    if "time" in out.columns:
        time = time_from_dt.fillna(out["time"].astype(str))
    else:
        time = time_from_dt.fillna("")

    time_under = pd.to_numeric(out["time_under"], errors="coerce").fillna(0).round().astype("int64")
    antenna = pd.to_numeric(out["antenna"], errors="coerce").astype("Int64")

    final = pd.DataFrame(
        {
            "index": range(len(out)),
            "date": date,
            "time": time,
            "antenna": antenna,
            "time_under": time_under,
            "animal_id": out["animal_id"].map(normalize_tag),
        }
    )
    return final


def write_old_style_file(df: pd.DataFrame, output_path: Path, sort_by_datetime: bool = True) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final = finalize_old_style_output(df, sort_by_datetime=sort_by_datetime)
    final.to_csv(output_path, sep="\t", header=False, index=False, lineterminator="\n")


def make_unique_path(path: Path) -> Path:
    if not path.exists():
        return path
    stem = path.stem
    suffix = path.suffix
    parent = path.parent
    i = 1
    while True:
        candidate = parent / f"{stem}_{i}{suffix}"
        if not candidate.exists():
            return candidate
        i += 1


def output_path_for_input(input_path: Path, output_dir: Path) -> Path:
    return make_unique_path(output_dir / f"{input_path.stem}_reformatted.txt")


# -----------------------------
# Format detection
# -----------------------------
def detect_format(path: Path) -> DetectedFile:
    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        return DetectedFile(path, UNKNOWN_FORMAT, "unsupported extension")

    lines = first_non_empty_lines(path, n=5)
    if not lines:
        return DetectedFile(path, UNKNOWN_FORMAT, "empty or unreadable")

    first = lines[0].strip().lstrip("\ufeff")
    header_norm = re.sub(r"\s+", "", first.lower())

    if all(token in header_norm for token in ["pctime", "channel", "boardtime", "rfidtag"]):
        return DetectedFile(path, OLD_ECOHAB_NEW_FW)

    # New DeepEcoHAB raw semicolon file: antenna;animal_id;hex_time;iso_datetime
    if ";" in first:
        parts = [p.strip() for p in first.split(";")]
        if len(parts) >= 4:
            if looks_like_int(parts[0]) and HEX_TAG_RE.match(parts[1]) and re.fullmatch(r"[0-9A-Fa-f]+", parts[2] or ""):
                if "T" in parts[3] or re.match(r"\d{4}-\d{2}-\d{2}", parts[3]):
                    return DetectedFile(path, NEW_DEEPECOHAB)

    # Old compressed format: index date time antenna time_under tag, usually tab-separated.
    parts_tab = first.split("\t") if "\t" in first else re.split(r"\s+", first)
    if len(parts_tab) >= 6:
        if looks_like_int(parts_tab[0]) and re.match(r"\d{4}\.\d{2}\.\d{2}", parts_tab[1]):
            if re.match(r"\d{2}:\d{2}:\d{2}", parts_tab[2]) and looks_like_int(parts_tab[3]):
                if looks_like_float(parts_tab[4]) and HEX_TAG_RE.match(parts_tab[5]):
                    return DetectedFile(path, OLD_ECOHAB_OLD_FW)

    return DetectedFile(path, UNKNOWN_FORMAT, "structure did not match known DAQ formats")


def scan_input_directory(input_dir: Path) -> List[DetectedFile]:
    if not input_dir.exists() or not input_dir.is_dir():
        return []
    detected: List[DetectedFile] = []
    for path in sorted(input_dir.iterdir()):
        if not path.is_file():
            continue
        if path.name.startswith("."):
            continue
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        detected.append(detect_format(path))
    return detected


# -----------------------------
# Tag and antenna map loading
# -----------------------------
def extract_tags_from_json_config(path: Path) -> List[str]:
    with path.open("r", encoding="utf-8-sig") as handle:
        data = json.load(handle)

    tags: List[str] = []

    animals = data.get("animals", {}) if isinstance(data, dict) else {}
    rows = animals.get("rows") if isinstance(animals, dict) else None

    if isinstance(rows, list):
        for row in rows:
            if isinstance(row, dict) and "tag_no" in row:
                tags.append(normalize_tag(row["tag_no"]))
    elif isinstance(rows, dict):
        for key, row in rows.items():
            if HEX_TAG_RE.match(str(key)):
                tags.append(normalize_tag(key))
            if isinstance(row, dict) and "tag_no" in row:
                tags.append(normalize_tag(row["tag_no"]))

    # Generic fallback: recursively collect any tag_no fields.
    def walk(obj: object) -> None:
        if isinstance(obj, dict):
            for key, value in obj.items():
                if str(key).lower() == "tag_no":
                    tags.append(normalize_tag(value))
                walk(value)
        elif isinstance(obj, list):
            for value in obj:
                walk(value)

    walk(data)
    return unique_preserve_order(tag for tag in tags if HEX_TAG_RE.match(tag))


def read_tag_list(path: Path) -> List[str]:
    if path.suffix.lower() == ".json":
        return extract_tags_from_json_config(path)

    text = path.read_text(encoding="utf-8-sig", errors="replace")
    raw_tokens = re.split(r"[,;\s]+", text)
    tags = []
    for token in raw_tokens:
        token = normalize_tag(token)
        if not token:
            continue
        if token.lower() in {"tag", "tag_no", "rfid", "rfid_tag", "animal_id"}:
            continue
        if HEX_TAG_RE.match(token):
            tags.append(token)
    return unique_preserve_order(tags)


def read_antenna_rename_map(path: Path) -> Dict[int, int]:
    text = path.read_text(encoding="utf-8-sig", errors="replace")
    mapping: Dict[int, int] = {}

    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        if "," in stripped:
            parts = [p.strip() for p in stripped.split(",")]
        elif ";" in stripped:
            parts = [p.strip() for p in stripped.split(";")]
        elif "\t" in stripped:
            parts = [p.strip() for p in stripped.split("\t")]
        else:
            parts = re.split(r"\s+", stripped)

        if len(parts) < 2:
            continue
        try:
            old = int(float(parts[0]))
            new = int(float(parts[1]))
        except ValueError:
            # Header row or invalid row.
            continue
        mapping[old] = new
    return mapping


# -----------------------------
# Reformatting logic
# -----------------------------
def estimate_board_time_scale_ms(datetime_series: pd.Series, board_time_series: pd.Series) -> float:
    """
    Estimate how many milliseconds one board-time unit represents.

    Old EcoHAB new-firmware files contain both PC time and board time. In some
    exports the board time is not exactly in milliseconds, so this keeps the GUI
    separation window in real milliseconds when possible.
    """
    tmp = pd.DataFrame(
        {
            "datetime": pd.to_datetime(datetime_series, errors="coerce"),
            "board_time": pd.to_numeric(board_time_series, errors="coerce"),
        }
    ).dropna()

    if len(tmp) < 3:
        return 1.0

    tmp = tmp.sort_values("datetime", kind="mergesort")
    pc_delta_ms = tmp["datetime"].diff().dt.total_seconds() * 1000.0
    board_delta = tmp["board_time"].diff()
    ratio = pc_delta_ms / board_delta
    ratio = ratio.replace([math.inf, -math.inf], pd.NA).dropna()
    ratio = ratio[(ratio > 0.001) & (ratio < 1000)]

    if ratio.empty:
        return 1.0
    scale = float(ratio.median())
    if not math.isfinite(scale) or scale <= 0:
        return 1.0
    return scale


def compress_raw_activations(raw: pd.DataFrame, separation_window_ms: int) -> pd.DataFrame:
    """
    Compress raw RFID reads into old-style antenna-activity periods.

    Corrected biologically-aware rule: compression is performed independently
    within each antenna stream.

    A new compressed period starts on a given antenna when:
      - this is the first read on that antenna, or
      - the detected animal changes on that same antenna, or
      - the same animal remains on that same antenna but the board-time gap is
        >= separation_window_ms, or
      - board time moved backwards, e.g. because of clock reset/wrap.

    Consequence:
      X on antenna 6, Y on antenna 6, X on antenna 6 -> X is split and Y is kept.
      X on antenna 6, Y on antenna 8, X on antenna 6 -> X is not split by Y.
    """
    required = {"animal_id", "antenna", "board_time_ms", "datetime"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Raw dataframe is missing columns: {', '.join(sorted(missing))}")

    df = raw.copy()
    df["animal_id"] = df["animal_id"].map(normalize_tag)
    df["antenna"] = pd.to_numeric(df["antenna"], errors="coerce").astype("Int64")
    df["board_time_ms"] = pd.to_numeric(df["board_time_ms"], errors="coerce")
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df.dropna(subset=["animal_id", "antenna", "board_time_ms", "datetime"])
    df = df[df["animal_id"] != ""].copy()

    if df.empty:
        return pd.DataFrame(columns=["datetime", "antenna", "time_under", "animal_id"])

    # Antenna-local stream order is the important part: detections on distant
    # antennas must not interrupt the current antenna's activity period.
    df = df.sort_values(["antenna", "datetime", "board_time_ms"], kind="mergesort").reset_index(drop=True)

    antenna_group = df.groupby("antenna", sort=False)
    df["prev_animal_id"] = antenna_group["animal_id"].shift()
    df["diff"] = antenna_group["board_time_ms"].diff()

    animal_changed_on_same_antenna = df["animal_id"] != df["prev_animal_id"]
    new_event = (
        df["diff"].isna()
        | animal_changed_on_same_antenna
        | (df["diff"] >= float(separation_window_ms))
        | (df["diff"] < 0)
    )

    # Event ids are independent per antenna stream. This preserves X/Y/X
    # interruptions on the same antenna but ignores Y events on other antennas.
    df["event_id"] = new_event.groupby(df["antenna"], sort=False).cumsum()
    df["internal_time"] = df["diff"].where(~new_event, 0).clip(lower=0).fillna(0)

    events = (
        df.groupby(["antenna", "event_id"], sort=False)
        .agg(
            datetime=("datetime", "first"),
            animal_id=("animal_id", "first"),
            time_under=("internal_time", "sum"),
        )
        .reset_index()
    )
    events = events[["datetime", "antenna", "time_under", "animal_id"]]
    return events


def convert_old_compressed_file(
    path: Path,
    tag_numbers: Sequence[str],
    keep_only_tags: bool,
    antenna_map: Dict[int, int],
) -> Tuple[pd.DataFrame, List[str]]:
    notes: List[str] = []
    names = ["old_index", "date", "time", "antenna", "time_under", "animal_id"]

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=names,
        dtype=str,
        engine="python",
        on_bad_lines="skip",
    )

    # Fallback for whitespace-separated old files.
    if df["date"].isna().all():
        df = pd.read_csv(
            path,
            sep=r"\s+",
            header=None,
            names=names,
            dtype=str,
            engine="python",
            on_bad_lines="skip",
        )

    before = len(df)
    df = apply_tag_filter(df, tag_numbers, keep_only_tags)
    if keep_only_tags:
        notes.append(f"tag filter: kept {len(df)} / {before} rows")

    df["antenna"] = replace_antenna_values(df["antenna"], antenna_map)
    df["time_under"] = pd.to_numeric(df["time_under"], errors="coerce").fillna(0)
    df["datetime"] = to_datetime_safe(df["date"].astype(str).str.replace(".", "-", regex=False) + " " + df["time"].astype(str))

    return df[["datetime", "date", "time", "antenna", "time_under", "animal_id"]], notes


def convert_new_deepecohab_file(
    path: Path,
    tag_numbers: Sequence[str],
    keep_only_tags: bool,
    antenna_map: Dict[int, int],
    separation_window_ms: int,
) -> Tuple[pd.DataFrame, List[str]]:
    notes: List[str] = []
    df = pd.read_csv(
        path,
        sep=";",
        header=None,
        names=["antenna", "animal_id", "board_time_hex", "datetime_raw"],
        dtype=str,
        engine="python",
        on_bad_lines="skip",
    )

    before = len(df)
    df = apply_tag_filter(df, tag_numbers, keep_only_tags)
    if keep_only_tags:
        notes.append(f"tag filter: kept {len(df)} / {before} raw reads")

    df["antenna"] = replace_antenna_values(df["antenna"], antenna_map)

    def parse_hex_to_ms(value: object) -> float:
        text = str(value).strip()
        if not re.fullmatch(r"[0-9A-Fa-f]+", text):
            return math.nan
        return float(int(text, 16) * 10)

    raw = pd.DataFrame(
        {
            "animal_id": df["animal_id"].map(normalize_tag),
            "antenna": df["antenna"],
            "board_time_ms": df["board_time_hex"].map(parse_hex_to_ms),
            "datetime": pd.to_datetime(df["datetime_raw"], errors="coerce"),
        }
    )
    compressed = compress_raw_activations(raw, separation_window_ms)
    notes.append(f"compressed {len(raw)} raw reads into {len(compressed)} periods")
    return compressed, notes


def convert_old_new_firmware_file(
    path: Path,
    tag_numbers: Sequence[str],
    keep_only_tags: bool,
    antenna_map: Dict[int, int],
    separation_window_ms: int,
) -> Tuple[pd.DataFrame, List[str]]:
    notes: List[str] = []
    df = pd.read_csv(path, sep=",", header=0, dtype=str, engine="python", on_bad_lines="skip")

    norm_cols = {re.sub(r"\s+", "", col).lower(): col for col in df.columns}
    required = {
        "pctime": "PC Time",
        "channel": "Channel",
        "boardtime": "Board Time",
        "rfidtag": "RFID Tag",
    }
    missing = [label for key, label in required.items() if key not in norm_cols]
    if missing:
        raise ValueError(f"Missing expected columns: {', '.join(missing)}")

    pc_col = norm_cols["pctime"]
    channel_col = norm_cols["channel"]
    board_col = norm_cols["boardtime"]
    tag_col = norm_cols["rfidtag"]

    before = len(df)
    tmp = pd.DataFrame(
        {
            "animal_id": df[tag_col].map(normalize_tag),
            "antenna": replace_antenna_values(df[channel_col], antenna_map),
            "board_time_raw": pd.to_numeric(df[board_col], errors="coerce"),
            "datetime": to_datetime_safe(df[pc_col]),
        }
    )

    tmp = apply_tag_filter(tmp, tag_numbers, keep_only_tags)
    if keep_only_tags:
        notes.append(f"tag filter: kept {len(tmp)} / {before} raw reads")

    scale_ms = estimate_board_time_scale_ms(tmp["datetime"], tmp["board_time_raw"])
    tmp["board_time_ms"] = tmp["board_time_raw"] * scale_ms
    if abs(scale_ms - 1.0) > 0.05:
        notes.append(f"estimated old-firmware board-time scale: {scale_ms:.4g} ms/unit")

    raw = tmp[["animal_id", "antenna", "board_time_ms", "datetime"]]
    compressed = compress_raw_activations(raw, separation_window_ms)
    notes.append(f"compressed {len(raw)} raw reads into {len(compressed)} periods")
    return compressed, notes


def convert_file_to_old_style(
    detected: DetectedFile,
    settings: ReformatSettings,
) -> Tuple[pd.DataFrame, List[str]]:
    if detected.fmt == OLD_ECOHAB_OLD_FW:
        return convert_old_compressed_file(
            detected.path,
            settings.tag_numbers,
            settings.keep_only_listed_tags,
            settings.antenna_map if settings.rename_antennas else {},
        )
    if detected.fmt == NEW_DEEPECOHAB:
        return convert_new_deepecohab_file(
            detected.path,
            settings.tag_numbers,
            settings.keep_only_listed_tags,
            settings.antenna_map if settings.rename_antennas else {},
            settings.separation_window_ms,
        )
    if detected.fmt == OLD_ECOHAB_NEW_FW:
        return convert_old_new_firmware_file(
            detected.path,
            settings.tag_numbers,
            settings.keep_only_listed_tags,
            settings.antenna_map if settings.rename_antennas else {},
            settings.separation_window_ms,
        )
    raise ValueError(f"Unsupported format: {detected.fmt}")


# -----------------------------
# Worker thread
# -----------------------------
class ReformatWorker(QObject):
    progress = Signal(int)
    message = Signal(str)
    finished = Signal(bool, str)

    def __init__(self, settings: ReformatSettings):
        super().__init__()
        self.settings = settings

    @Slot()
    def run(self) -> None:
        try:
            detected = scan_input_directory(self.settings.input_dir)
            supported = [d for d in detected if d.fmt != UNKNOWN_FORMAT]
            if not supported:
                self.finished.emit(False, "No supported DAQ files were found in the input directory.")
                return

            self.settings.output_dir.mkdir(parents=True, exist_ok=True)
            self.message.emit(f"Supported files to process: {len(supported)}")
            self.message.emit(f"Separation window: >= {self.settings.separation_window_ms} ms")

            all_outputs: List[pd.DataFrame] = []
            written_files: List[Path] = []
            errors: List[str] = []
            n = len(supported)

            for i, item in enumerate(supported, start=1):
                self.message.emit(f"\n[{i}/{n}] {item.path.name} — {item.fmt}")
                try:
                    converted, notes = convert_file_to_old_style(item, self.settings)
                    for note in notes:
                        self.message.emit(f"  {note}")
                    if self.settings.stitch_files:
                        converted = converted.copy()
                        converted["source_file"] = item.path.name
                        all_outputs.append(converted)
                    else:
                        out_path = output_path_for_input(item.path, self.settings.output_dir)
                        write_old_style_file(converted, out_path, sort_by_datetime=True)
                        written_files.append(out_path)
                        self.message.emit(f"  wrote: {out_path}")
                except Exception as exc:
                    errors.append(f"{item.path.name}: {exc}")
                    self.message.emit(f"  ERROR: {exc}")

                self.progress.emit(int(i / n * 90))

            if self.settings.stitch_files:
                if not all_outputs:
                    self.finished.emit(False, "No files were converted successfully.")
                    return
                stitched = pd.concat(all_outputs, ignore_index=True)
                stitched_name = f"{self.settings.input_dir.name}_stitched_reformatted.txt"
                out_path = make_unique_path(self.settings.output_dir / stitched_name)
                write_old_style_file(stitched, out_path, sort_by_datetime=True)
                written_files.append(out_path)
                self.message.emit(f"\nstitched output written: {out_path}")

            self.progress.emit(100)
            summary = f"Done. Wrote {len(written_files)} output file(s)."
            if errors:
                summary += f" {len(errors)} file(s) failed."
                self.message.emit("\nFailed files:")
                for err in errors:
                    self.message.emit(f"  - {err}")
            self.finished.emit(len(written_files) > 0, summary)
        except Exception as exc:
            self.finished.emit(False, str(exc))



# -----------------------------
# DeepEcoHAB visual theme
# -----------------------------
def apply_deepecohab_theme(app: QApplication) -> None:
    """Use the same dark style as the New DeepEcoHAB RFID DAQ GUI."""
    try:
        app.setStyle("Fusion")
    except Exception:
        pass

    pal = QPalette()
    pal.setColor(QPalette.Window, QColor("#0f1115"))
    pal.setColor(QPalette.WindowText, QColor("#e6e6e6"))
    pal.setColor(QPalette.Base, QColor("#12151c"))
    pal.setColor(QPalette.AlternateBase, QColor("#0f1115"))
    pal.setColor(QPalette.ToolTipBase, QColor("#0f1115"))
    pal.setColor(QPalette.ToolTipText, QColor("#e6e6e6"))
    pal.setColor(QPalette.Text, QColor("#e6e6e6"))
    pal.setColor(QPalette.Button, QColor("#151a23"))
    pal.setColor(QPalette.ButtonText, QColor("#e6e6e6"))
    pal.setColor(QPalette.BrightText, QColor("#ffffff"))
    pal.setColor(QPalette.Highlight, QColor("#2d6cdf"))
    pal.setColor(QPalette.HighlightedText, QColor("#ffffff"))
    app.setPalette(pal)

    app.setStyleSheet("""
        QWidget { background: #0f1115; color: #e6e6e6; font-family: Segoe UI, Arial; }
        QToolTip { color: #e6e6e6; background-color: #0f1115; border: 1px solid #2b2f3a; }
        QGroupBox { border: 1px solid #2b2f3a; border-radius: 8px; margin-top: 12px; padding: 10px; background: #0f1115; font-size: 10pt; }
        QGroupBox::title { subcontrol-origin: margin; left: 10px; padding: 0 4px; color: #e6e6e6; font-weight: 600; }
        QLineEdit, QSpinBox { background: #12151c; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 6px; padding: 5px 7px; min-height: 24px; }
        QPlainTextEdit { background: #12151c; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 6px; padding: 6px; }
        QPushButton { background: #1a2230; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 8px; padding: 7px 12px; min-height: 28px; font-weight: 600; }
        QPushButton:hover { background: #22304a; border: 1px solid #2d6cdf; }
        QPushButton:pressed { background: #2d6cdf; color: #ffffff; }
        QPushButton:disabled { background: #11151d; color: #7c7c7c; }
        QCheckBox { color: #e6e6e6; spacing: 8px; }
        QProgressBar { background: #12151c; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 6px; text-align: center; }
        QProgressBar::chunk { background: #2d6cdf; border-radius: 5px; }
        QMessageBox { background: #0f1115; color: #e6e6e6; }
    """)


# -----------------------------
# GUI
# -----------------------------
class MainWindow(QMainWindow):
    def __init__(self) -> None:
        super().__init__()
        self.setWindowTitle("DeepEcoHAB RFID dataframe reformatter")
        self.resize(1380, 880)
        self.setMinimumSize(1050, 700)

        self.detected_files: List[DetectedFile] = []
        self.tag_numbers: List[str] = []
        self.antenna_map: Dict[int, int] = {}
        self.thread: Optional[QThread] = None
        self.worker: Optional[ReformatWorker] = None

        root = QWidget()
        self.setCentralWidget(root)
        main_layout = QVBoxLayout(root)
        main_layout.setContentsMargins(14, 12, 14, 14)
        main_layout.setSpacing(10)

        title = QLabel("DeepEcoHAB / EcoHAB RFID dataframe reformatter")
        title.setStyleSheet("font-size: 18px; font-weight: 700; color: #e6e6e6;")
        subtitle = QLabel("Convert new-board and old-board RFID outputs into downstream-analysis-ready EcoHAB tables.")
        subtitle.setStyleSheet("color: #94a3b8; font-size: 10pt;")
        main_layout.addWidget(title)
        main_layout.addWidget(subtitle)

        grid = QGridLayout()
        grid.setColumnStretch(0, 1)
        grid.setColumnStretch(1, 1)
        grid.setRowStretch(0, 1)
        grid.setRowStretch(1, 2)
        grid.setRowStretch(2, 1)
        grid.setHorizontalSpacing(14)
        grid.setVerticalSpacing(10)
        main_layout.addLayout(grid, stretch=1)

        # ------------------------------------------------------------
        # LEFT COLUMN, ROW 0: input/output controls
        # ------------------------------------------------------------
        io_group = QGroupBox("Input / output")
        io_layout = QFormLayout(io_group)
        io_layout.setLabelAlignment(Qt.AlignRight)
        io_layout.setFormAlignment(Qt.AlignTop)
        io_layout.setHorizontalSpacing(10)
        io_layout.setVerticalSpacing(8)

        self.input_dir_edit = QLineEdit()
        self.input_dir_edit.setMinimumWidth(360)
        self.input_browse_btn = QPushButton("Browse…")
        self.input_browse_btn.setMinimumWidth(105)
        input_row = QHBoxLayout()
        input_row.setContentsMargins(0, 0, 0, 0)
        input_row.setSpacing(8)
        input_row.addWidget(self.input_dir_edit, stretch=1)
        input_row.addWidget(self.input_browse_btn)
        io_layout.addRow("Dataframes directory", input_row)

        self.output_dir_edit = QLineEdit()
        self.output_dir_edit.setMinimumWidth(360)
        self.output_browse_btn = QPushButton("Browse…")
        self.output_browse_btn.setMinimumWidth(105)
        output_row = QHBoxLayout()
        output_row.setContentsMargins(0, 0, 0, 0)
        output_row.setSpacing(8)
        output_row.addWidget(self.output_dir_edit, stretch=1)
        output_row.addWidget(self.output_browse_btn)
        io_layout.addRow("Write reformatted files to", output_row)

        self.stitch_checkbox = QCheckBox("Stitch detected files into one datetime-sorted output file")
        io_layout.addRow("", self.stitch_checkbox)

        self.check_btn = QPushButton("Check dataframes")
        self.check_btn.setMinimumHeight(36)
        io_layout.addRow("", self.check_btn)
        grid.addWidget(io_group, 0, 0)

        # ------------------------------------------------------------
        # RIGHT COLUMN, ROW 0: true RFID tag controls
        # ------------------------------------------------------------
        tag_group = QGroupBox("True RFID tag numbers")
        tag_group.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        tag_layout = QVBoxLayout(tag_group)
        tag_layout.setContentsMargins(8, 10, 8, 8)
        tag_layout.setSpacing(8)
        tag_form = QFormLayout()
        tag_form.setLabelAlignment(Qt.AlignRight)
        tag_form.setVerticalSpacing(8)

        self.tag_file_edit = QLineEdit()
        self.tag_file_edit.setMinimumWidth(300)
        self.tag_browse_btn = QPushButton("Browse…")
        self.tag_browse_btn.setMinimumWidth(105)
        tag_row = QHBoxLayout()
        tag_row.setContentsMargins(0, 0, 0, 0)
        tag_row.setSpacing(8)
        tag_row.addWidget(self.tag_file_edit, stretch=1)
        tag_row.addWidget(self.tag_browse_btn)
        tag_form.addRow("CSV/TXT tag list or JSON config", tag_row)

        self.read_tags_btn = QPushButton("Read tag list")
        self.read_tags_btn.setMinimumHeight(34)
        tag_form.addRow("", self.read_tags_btn)

        self.keep_tags_checkbox = QCheckBox("Keep only listed tag numbers")
        tag_form.addRow("", self.keep_tags_checkbox)
        tag_layout.addLayout(tag_form)

        self.tag_output = QPlainTextEdit()
        self.tag_output.setReadOnly(True)
        self.tag_output.setMinimumHeight(90)
        self.tag_output.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.tag_output.setPlaceholderText("Read tag numbers will appear here.")
        tag_layout.addWidget(self.tag_output, stretch=1)
        grid.addWidget(tag_group, 0, 1)

        # ------------------------------------------------------------
        # LEFT COLUMN, ROW 1: detection/check output
        # ------------------------------------------------------------
        check_group = QGroupBox("Dataframe check output")
        check_group.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        check_layout = QVBoxLayout(check_group)
        check_layout.setContentsMargins(8, 10, 8, 8)
        self.check_output = QPlainTextEdit()
        self.check_output.setReadOnly(True)
        self.check_output.setMinimumHeight(140)
        self.check_output.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.check_output.setPlaceholderText("Detected dataframe formats will appear here.")
        check_layout.addWidget(self.check_output, stretch=1)
        grid.addWidget(check_group, 1, 0)

        # ------------------------------------------------------------
        # RIGHT COLUMN, ROW 1: antenna renaming controls
        # ------------------------------------------------------------
        antenna_group = QGroupBox("Antenna renaming")
        antenna_group.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        antenna_layout = QVBoxLayout(antenna_group)
        antenna_layout.setContentsMargins(8, 10, 8, 8)
        antenna_layout.setSpacing(8)
        antenna_form = QFormLayout()
        antenna_form.setLabelAlignment(Qt.AlignRight)
        antenna_form.setVerticalSpacing(8)

        self.rename_antennas_checkbox = QCheckBox("Rename antennas")
        antenna_form.addRow("", self.rename_antennas_checkbox)

        self.antenna_file_edit = QLineEdit()
        self.antenna_file_edit.setMinimumWidth(300)
        self.antenna_browse_btn = QPushButton("Browse…")
        self.antenna_browse_btn.setMinimumWidth(105)
        antenna_file_row = QHBoxLayout()
        antenna_file_row.setContentsMargins(0, 0, 0, 0)
        antenna_file_row.setSpacing(8)
        antenna_file_row.addWidget(self.antenna_file_edit, stretch=1)
        antenna_file_row.addWidget(self.antenna_browse_btn)
        antenna_form.addRow("Antenna rename CSV", antenna_file_row)

        self.read_antenna_btn = QPushButton("Read antenna rename list")
        self.read_antenna_btn.setMinimumHeight(34)
        antenna_form.addRow("", self.read_antenna_btn)
        antenna_layout.addLayout(antenna_form)

        self.antenna_output = QPlainTextEdit()
        self.antenna_output.setReadOnly(True)
        self.antenna_output.setMinimumHeight(90)
        self.antenna_output.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.antenna_output.setPlaceholderText("Antenna rename mapping will appear here.")
        antenna_layout.addWidget(self.antenna_output, stretch=1)
        grid.addWidget(antenna_group, 1, 1)

        # ------------------------------------------------------------
        # LEFT COLUMN, ROW 2: compression window
        # ------------------------------------------------------------
        time_group = QGroupBox("Compression window")
        time_layout = QFormLayout(time_group)
        time_layout.setLabelAlignment(Qt.AlignRight)
        time_layout.setVerticalSpacing(8)
        self.change_window_checkbox = QCheckBox("Change time under antenna separation window")
        time_layout.addRow("", self.change_window_checkbox)

        self.window_spin = QSpinBox()
        self.window_spin.setRange(1, 10000)
        self.window_spin.setValue(90)
        self.window_spin.setEnabled(False)
        time_layout.addRow("Board-time gap threshold (ms)", self.window_spin)

        default_label = QLabel("By default, compressed detection periods are separated by gaps ≥ 90 ms.")
        default_label.setWordWrap(True)
        default_label.setStyleSheet("color: #94a3b8;")
        time_layout.addRow("", default_label)
        grid.addWidget(time_group, 2, 0)

        # ------------------------------------------------------------
        # RIGHT COLUMN, ROW 2: run/reformat controls
        # ------------------------------------------------------------
        run_group = QGroupBox("Reformat")
        run_group.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        run_layout = QVBoxLayout(run_group)
        run_layout.setContentsMargins(8, 10, 8, 8)
        run_layout.setSpacing(8)
        self.reformat_btn = QPushButton("Reformat")
        self.reformat_btn.setMinimumHeight(44)
        self.progress_bar = QProgressBar()
        self.progress_bar.setRange(0, 100)
        self.status_label = QLabel("Ready.")
        self.status_label.setStyleSheet("color: #cbd5e1;")
        self.run_log = QPlainTextEdit()
        self.run_log.setReadOnly(True)
        self.run_log.setMinimumHeight(120)
        self.run_log.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.run_log.setPlaceholderText("Reformatting log will appear here.")
        run_layout.addWidget(self.reformat_btn)
        run_layout.addWidget(self.progress_bar)
        run_layout.addWidget(self.status_label)
        run_layout.addWidget(self.run_log, stretch=1)
        grid.addWidget(run_group, 2, 1)

        # Connections
        self.input_browse_btn.clicked.connect(self.browse_input_dir)
        self.output_browse_btn.clicked.connect(self.browse_output_dir)
        self.check_btn.clicked.connect(self.check_dataframes)
        self.tag_browse_btn.clicked.connect(self.browse_tag_file)
        self.read_tags_btn.clicked.connect(self.read_tag_list_clicked)
        self.rename_antennas_checkbox.toggled.connect(self.set_antenna_controls_enabled)
        self.antenna_browse_btn.clicked.connect(self.browse_antenna_file)
        self.read_antenna_btn.clicked.connect(self.read_antenna_map_clicked)
        self.change_window_checkbox.toggled.connect(self.window_spin.setEnabled)
        self.reformat_btn.clicked.connect(self.reformat_clicked)

        self.set_antenna_controls_enabled(False)

    def browse_input_dir(self) -> None:
        directory = QFileDialog.getExistingDirectory(self, "Select dataframe directory")
        if directory:
            self.input_dir_edit.setText(directory)
            if not self.output_dir_edit.text().strip():
                self.output_dir_edit.setText(directory)

    def browse_output_dir(self) -> None:
        directory = QFileDialog.getExistingDirectory(self, "Select output directory")
        if directory:
            self.output_dir_edit.setText(directory)

    def browse_tag_file(self) -> None:
        path, _ = QFileDialog.getOpenFileName(
            self,
            "Select tag list or DeepEcoHAB config",
            "",
            "Tag/config files (*.csv *.txt *.tsv *.json);;All files (*)",
        )
        if path:
            self.tag_file_edit.setText(path)

    def browse_antenna_file(self) -> None:
        path, _ = QFileDialog.getOpenFileName(
            self,
            "Select antenna rename CSV",
            "",
            "CSV/TXT files (*.csv *.txt *.tsv);;All files (*)",
        )
        if path:
            self.antenna_file_edit.setText(path)

    def set_antenna_controls_enabled(self, enabled: bool) -> None:
        self.antenna_file_edit.setEnabled(enabled)
        self.antenna_browse_btn.setEnabled(enabled)
        self.read_antenna_btn.setEnabled(enabled)
        self.antenna_output.setEnabled(enabled)

    def check_dataframes(self) -> None:
        input_dir = Path(self.input_dir_edit.text().strip())
        if not input_dir.is_dir():
            QMessageBox.warning(self, "Input directory", "Please select a valid dataframe directory.")
            return

        self.detected_files = scan_input_directory(input_dir)
        counts: Dict[str, int] = {}
        for item in self.detected_files:
            counts[item.fmt] = counts.get(item.fmt, 0) + 1

        supported_count = sum(1 for item in self.detected_files if item.fmt != UNKNOWN_FORMAT)
        lines = [f"Files checked: {len(self.detected_files)}", f"Supported DAQ files detected: {supported_count}", ""]
        for fmt in [NEW_DEEPECOHAB, OLD_ECOHAB_NEW_FW, OLD_ECOHAB_OLD_FW, UNKNOWN_FORMAT]:
            lines.append(f"{fmt}: {counts.get(fmt, 0)}")

        lines.append("\nDetected files:")
        for item in self.detected_files:
            reason = f" ({item.reason})" if item.reason else ""
            lines.append(f"- {item.path.name}: {item.fmt}{reason}")
        self.check_output.setPlainText("\n".join(lines))

    def read_tag_list_clicked(self) -> None:
        path = Path(self.tag_file_edit.text().strip())
        if not path.is_file():
            QMessageBox.warning(self, "Tag list", "Please select a valid CSV/TXT tag list or JSON config file.")
            return
        try:
            self.tag_numbers = read_tag_list(path)
        except Exception as exc:
            QMessageBox.critical(self, "Tag list", f"Could not read tag list:\n{exc}")
            return

        lines = [f"Read {len(self.tag_numbers)} tag number(s):"]
        lines.extend(self.tag_numbers)
        self.tag_output.setPlainText("\n".join(lines))

    def read_antenna_map_clicked(self) -> None:
        path = Path(self.antenna_file_edit.text().strip())
        if not path.is_file():
            QMessageBox.warning(self, "Antenna rename", "Please select a valid antenna rename CSV/TXT file.")
            return
        try:
            self.antenna_map = read_antenna_rename_map(path)
        except Exception as exc:
            QMessageBox.critical(self, "Antenna rename", f"Could not read antenna rename list:\n{exc}")
            return

        lines = [f"Read {len(self.antenna_map)} antenna rename rule(s):"]
        for old, new in sorted(self.antenna_map.items()):
            lines.append(f"{old} → {new}")
        self.antenna_output.setPlainText("\n".join(lines))

    def build_settings(self) -> Optional[ReformatSettings]:
        input_dir = Path(self.input_dir_edit.text().strip())
        output_dir = Path(self.output_dir_edit.text().strip())

        if not input_dir.is_dir():
            QMessageBox.warning(self, "Input directory", "Please select a valid dataframe directory.")
            return None
        if not output_dir:
            QMessageBox.warning(self, "Output directory", "Please select an output directory.")
            return None

        if self.keep_tags_checkbox.isChecked() and not self.tag_numbers:
            QMessageBox.warning(
                self,
                "Tag filter",
                "'Keep only listed tag numbers' is enabled, but no tag list has been read.",
            )
            return None

        if self.rename_antennas_checkbox.isChecked() and not self.antenna_map:
            QMessageBox.warning(
                self,
                "Antenna rename",
                "'Rename antennas' is enabled, but no antenna rename list has been read.",
            )
            return None

        return ReformatSettings(
            input_dir=input_dir,
            output_dir=output_dir,
            stitch_files=self.stitch_checkbox.isChecked(),
            keep_only_listed_tags=self.keep_tags_checkbox.isChecked(),
            tag_numbers=self.tag_numbers,
            rename_antennas=self.rename_antennas_checkbox.isChecked(),
            antenna_map=self.antenna_map,
            separation_window_ms=int(self.window_spin.value()),
        )

    def reformat_clicked(self) -> None:
        settings = self.build_settings()
        if settings is None:
            return

        self.progress_bar.setValue(0)
        self.status_label.setText("Reformatting…")
        self.run_log.clear()
        self.reformat_btn.setEnabled(False)

        self.thread = QThread(self)
        self.worker = ReformatWorker(settings)
        self.worker.moveToThread(self.thread)

        self.thread.started.connect(self.worker.run)
        self.worker.progress.connect(self.progress_bar.setValue)
        self.worker.message.connect(self.append_run_log)
        self.worker.finished.connect(self.reformat_finished)
        self.worker.finished.connect(self.thread.quit)
        self.worker.finished.connect(self.worker.deleteLater)
        self.thread.finished.connect(self.thread.deleteLater)
        self.thread.start()

    @Slot(str)
    def append_run_log(self, text: str) -> None:
        self.run_log.appendPlainText(text)

    @Slot(bool, str)
    def reformat_finished(self, success: bool, message: str) -> None:
        self.reformat_btn.setEnabled(True)
        self.status_label.setText(message)
        if success:
            self.progress_bar.setValue(100)
        else:
            QMessageBox.warning(self, "Reformatting", message)
        self.thread = None
        self.worker = None


def main() -> int:
    app = QApplication(sys.argv)
    apply_deepecohab_theme(app)
    app.setApplicationName("DeepEcoHAB RFID dataframe reformatter")
    window = MainWindow()
    window.show()
    return app.exec()


if __name__ == "__main__":
    raise SystemExit(main())
